<a href="https://colab.research.google.com/github/pratham414/AI-ML/blob/main/AI2may3Q.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
pip install pyspellChecker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 34.9 MB/s eta 0:00:00


In [18]:
# @title Default title text
import pandas as pd
import numpy as np
import re
import nltk
from nltk.tokenize import word_tokenize as wt
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from spellchecker import SpellChecker
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

dataset = pd.read_csv("/content/drive/My Drive/spam.csv", encoding="latin1")

stemmer = PorterStemmer()#LancasterStemmer(), SnowballStemmer(),WordNetLemmatizer()
spell = SpellChecker()#pip install pyspellchecker
data = []
print("No. of rows ",dataset.shape[0])#displays total rows
#print("No.of columns ",dataset.shape[1])#displays total columns
for i in range(dataset.shape[0]): ##accessing each row 5572 rows in total
     sms = dataset.iloc[i, 1] #i=row no. and 1 is column 1
     sms = re.sub('[^A-Za-z]', ' ', sms) #removing all symbols other than alphabets
     sms = sms.lower()
     tokenized_sms = wt(sms)#word tokenization

     # stopword removal and stemming
     sms_processed = []
     for word in tokenized_sms:
         if word not in set(stopwords.words('english')):
             sms_processed.append((stemmer.stem(word)))
             #sms_processed.append(spell.correction((stemmer.stem(word))))

     sms_text =" ".join(sms_processed)
     data.append(sms_text)#data is a list, where each list variable containing sentence

 # creating the feature matrix
bow = CountVectorizer(max_features=1000)
#matrix = CountVectorizer()
X = bow.fit_transform(data).toarray() # input data used for learning and building BOW model
#print(X)
y = dataset.iloc[:, 0]#filters all rows : from 0th column "output labels"
#df['msg']= df['spamham'].map({'ham': 0, 'spam': 1})
y=y.map({'ham': 0, 'spam': 1})
# split train and test data
X_train, X_test, y_train, y_test = train_test_split(X, y) #from sklearn.model_selection

# Naive Bayes
#model = BernoulliNB() # if X data is majorly binary
#model = MultinomialNB()# For multiclass classificaiton
model = GaussianNB() #suitable if data is continuous and assumed to be on Gaussian distribution
model.fit(X_train, y_train)#model training

# predict class
y_pred = model.predict(X_test)
print("test data result y_pred \n",y_pred)

################
 # Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix \n",cm)
cr = classification_report(y_test, y_pred)
print("Classificaiton Report is \n",cr)
accuracy = accuracy_score(y_test, y_pred)
print("last accuracy ",accuracy)
##############Testing with Actual data
#myvector=bow.transform(["Enjoy Students holiday today"]).toarray()
myvector=bow.transform(["Students regular lecture today"]).toarray()
y_result=model.predict(myvector)
print("y_result ",y_result)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


No. of rows  5572
test data result y_pred 
 [0 1 1 ... 1 1 0]
Confusion Matrix 
 [[911 290]
 [ 14 178]]
Classificaiton Report is 
               precision    recall  f1-score   support

           0       0.98      0.76      0.86      1201
           1       0.38      0.93      0.54       192

    accuracy                           0.78      1393
   macro avg       0.68      0.84      0.70      1393
weighted avg       0.90      0.78      0.81      1393

last accuracy  0.7817659727207465
y_result  [1]


In [19]:
from sklearn.svm import SVC

# Initialize SVM model
svm_model = SVC(kernel='linear')  # Linear kernel works well for text classification
svm_model.fit(X_train, y_train)

# Predict using SVM
y_pred_svm = svm_model.predict(X_test)

# Evaluate SVM model
print("SVM Confusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("SVM Classification Report:\n", classification_report(y_test, y_pred_svm))
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))

SVM Confusion Matrix:
 [[1192    9]
 [  21  171]]
SVM Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.99      1201
           1       0.95      0.89      0.92       192

    accuracy                           0.98      1393
   macro avg       0.97      0.94      0.95      1393
weighted avg       0.98      0.98      0.98      1393

SVM Accuracy: 0.9784637473079684


In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

# Define ANN model
model = Sequential([
    Input(shape=(X_train.shape[1],)),  # Input layer
    Dense(128, activation='sigmoid'),  # Hidden layer with 128 neurons
    Dense(64, activation='sigmoid'),   # Hidden layer with 64 neurons
    Dense(32, activation='sigmoid'),   # Hidden layer with 32 neurons
    Dense(16, activation='sigmoid'),   # Hidden layer with 16 neurons
    Dense(1, activation='sigmoid')     # Output layer for binary classification
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test))

# Predict on test data
y_pred_ann = model.predict(X_test)
y_pred_ann = (y_pred_ann > 0.5).astype(int)  # Convert probabilities to binary labels

# Predict a custom message
test_sms = ["Congratulations! You've won a free trip. Claim now!"]
X_new = bow.transform(test_sms).toarray()  # Convert to feature vector
y_result_ann = model.predict(X_new)
y_result_ann = (y_result_ann > 0.5).astype(int)

print("Predicted Class (0 = Ham, 1 = Spam):", y_result_ann[0])

# Evaluate model accuracy on training data
train_loss, train_accuracy = model.evaluate(X_train, y_train)
print(f"Training Accuracy: {train_accuracy}")

# Evaluate model accuracy on testing data
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Testing Accuracy: {test_accuracy}")

Epoch 1/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.8861 - loss: 0.3365 - val_accuracy: 0.9835 - val_loss: 0.0894
Epoch 2/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9876 - loss: 0.0647 - val_accuracy: 0.9828 - val_loss: 0.0771
Epoch 3/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9946 - loss: 0.0309 - val_accuracy: 0.9742 - val_loss: 0.0994
Epoch 4/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9909 - loss: 0.0398 - val_accuracy: 0.9821 - val_loss: 0.0760
Epoch 5/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9959 - loss: 0.0175 - val_accuracy: 0.9821 - val_loss: 0.0718
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Predicted Class (0 = Ham, 1 = Spam): [1]
131/131 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9976 - loss: 0.0148
Training Accuracy: 0.9988035559654236
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9813 - loss: 0.0744
Testing Accuracy: 0.9820531010627747


In [23]:
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDClassifier

rbf_feature = RBFSampler(gamma=1, random_state=42)
X_train_rbf = rbf_feature.fit_transform(X_train)
X_test_rbf = rbf_feature.transform(X_test)

# Train an RBF-based classifier (can be logistic regression, SVM, etc.)
rbf_model = SGDClassifier(loss='log_loss', max_iter=500)
rbf_model.fit(X_train_rbf, y_train)

# Predict using RBF model
y_pred_rbf = rbf_model.predict(X_test_rbf)

# Evaluate RBF model
print("RBF Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rbf))
print("RBF Classification Report:\n", classification_report(y_test, y_pred_rbf))
print("RBF Accuracy:", accuracy_score(y_test, y_pred_rbf))

RBF Confusion Matrix:
 [[1196    5]
 [ 180   12]]
RBF Classification Report:
               precision    recall  f1-score   support

           0       0.87      1.00      0.93      1201
           1       0.71      0.06      0.11       192

    accuracy                           0.87      1393
   macro avg       0.79      0.53      0.52      1393
weighted avg       0.85      0.87      0.82      1393

RBF Accuracy: 0.8671931083991385
